<a href="https://colab.research.google.com/github/stacydorce06/COT20000125-repo/blob/main/Section_3_Queueless_code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Conceptual Python Script for QueueLess Core Logic (for Google Colab)
*This script simulates the backend logic of QueueLess, focusing on managing queues, estimating wait times, and notifying users.*



In [11]:
import time
import random
from datetime import datetime, timedelta
import math # For ceil function

# --- Configuration ---
# Simulate how often wait times are "updated" by businesses (in seconds)
WAIT_TIME_UPDATE_INTERVAL = 60
# Base average time a single customer takes for a generic service (in minutes)
BASE_AVG_SERVICE_TIME_PER_CUSTOMER = 5

# --- Data Structures (Simulated Database) ---
# Stores business information
# key: business_id, value: {name, type, location, service_capacity, current_queue, avg_customer_time, last_queue_update}
# current_queue: List of (user_id, join_time, estimated_service_start_time)
businesses = {
    "pharmacy_A": {
        "name": "QuickRx Pharmacy",
        "type": "Pharmacy",
        "location": "123 Main St",
        "service_capacity": 2, # Number of service desks/staff available
        "current_queue": [], # List of (user_id, join_time, estimated_service_start_time)
        "avg_customer_time": 8, # Specific average time for this business (minutes per customer)
        "last_queue_update": datetime.now() # To simulate when the queue data was last updated
    },
    "coffee_shop_B": {
        "name": "Morning Brew",
        "type": "Coffee Shop",
        "location": "456 Oak Ave",
        "service_capacity": 3,
        "current_queue": [],
        "avg_customer_time": 3,
        "last_queue_update": datetime.now()
    },
    "clinic_C": {
        "name": "Wellness Clinic",
        "type": "Clinic",
        "location": "789 Health Blvd",
        "service_capacity": 1,
        "current_queue": [],
        "avg_customer_time": 15,
        "last_queue_update": datetime.now()
    },
    "barber_D": {
        "name": "Classic Cuts",
        "type": "Barbershop",
        "location": "101 High St",
        "service_capacity": 2,
        "current_queue": [],
        "avg_customer_time": 20,
        "last_queue_update": datetime.now()
    }
}

# Stores user information (for notifications)
# key: user_id, value: {name, phone_number, current_business_id, estimated_arrival_time}
users = {}

# --- Helper Functions ---

def calculate_estimated_wait_time(business_id):
    """
    Calculates the estimated wait time for a business.
    This is a simplified model. A real system would use more sophisticated algorithms
    based on historical data, staff availability, service type, etc.
    """
    business = businesses.get(business_id)
    if not business:
        return "N/A"
    queue_length = len(business["current_queue"])
    service_capacity = business["service_capacity"]
    avg_customer_time = business["avg_customer_time"]

    if service_capacity == 0: # Avoid division by zero
        return float('inf') # Infinite wait

    # The number of people 'ahead' of a new customer who are currently waiting
    # and who cannot be served immediately by available capacity.
    people_ahead_waiting = max(0, queue_length - service_capacity)

    # Estimated wait time based on the number of people ahead who are waiting
    # and the average time it takes to serve one customer, divided by capacity.
    estimated_wait_minutes = math.ceil((people_ahead_waiting * avg_customer_time) / service_capacity)

    return estimated_wait_minutes

def join_queue(user_id, business_id):
    """
    Allows a user to join the queue for a specific business.
    """
    if business_id not in businesses:
        return f"Error: Business '{business_id}' not found.", None

    if user_id in users and users[user_id].get('current_business_id') == business_id:
        return f"User '{user_id}' is already in the queue for '{businesses[business_id]['name']}'.", None
    elif user_id in users and users[user_id].get('current_business_id') is not None:
        return f"User '{user_id}' is already in the queue for another business: '{businesses[users[user_id]['current_business_id']]['name']}'.", None

    join_time = datetime.now()
    business = businesses[business_id]

    # Update user information
    if user_id not in users:
        users[user_id] = {'name': f'User {user_id}', 'phone_number': '555-123-4567', 'current_business_id': None, 'estimated_arrival_time': None}
    users[user_id]['current_business_id'] = business_id

    # Calculate initial estimated service start time for this user
    # This is a placeholder; a real system would dynamically update this for all queue members.
    current_queue_length = len(business['current_queue'])
    service_capacity = business['service_capacity']
    avg_customer_time = business['avg_customer_time']

    # Calculate position in terms of service slots
    # If capacity is 2, first two people are at position 0, next two at position 1, etc.
    # This simplifies calculation for initial service start time
    if service_capacity > 0:
        queue_position_multiplier = math.floor(current_queue_length / service_capacity)
        estimated_service_start_time = join_time + timedelta(minutes=queue_position_multiplier * avg_customer_time)
    else:
        estimated_service_start_time = join_time + timedelta(minutes=float('inf')) # Effectively infinite

    # Add user to the business's queue
    business['current_queue'].append((user_id, join_time, estimated_service_start_time))

    # Update user's estimated arrival time (this will be refined by real-time updates)
    users[user_id]['estimated_arrival_time'] = estimated_service_start_time

    estimated_wait = calculate_estimated_wait_time(business_id)
    print(f"User '{user_id}' joined the queue for '{business['name']}'. Estimated wait: {estimated_wait} minutes.")
    return f"Successfully joined '{business['name']}'.", estimated_wait

def leave_queue(user_id, business_id):
    """
    Allows a user to leave the queue for a specific business.
    """
    if business_id not in businesses:
        return f"Error: Business '{business_id}' not found."

    business = businesses[business_id]
    user_in_queue = False
    for i, (q_user_id, _, _) in enumerate(business['current_queue']):
        if q_user_id == user_id:
            # Remove the user from the queue
            business['current_queue'].pop(i)
            user_in_queue = True
            break

    if not user_in_queue:
        return f"Error: User '{user_id}' is not in the queue for '{business['name']}'."

    # Update user information to reflect they are no longer in a queue
    if user_id in users:
        users[user_id]['current_business_id'] = None
        users[user_id]['estimated_arrival_time'] = None

    # Recalculate estimated service start times for remaining users in this queue
    service_capacity = business['service_capacity']
    avg_customer_time = business['avg_customer_time']
    for i, (q_user_id, join_time, _) in enumerate(business['current_queue']):
        if service_capacity > 0:
            queue_position_multiplier = math.floor(i / service_capacity)
            recalculated_est_start_time = join_time + timedelta(minutes=queue_position_multiplier * avg_customer_time)
        else:
            recalculated_est_start_time = join_time + timedelta(minutes=float('inf'))
        business['current_queue'][i] = (q_user_id, join_time, recalculated_est_start_time)
        # Also update the user's estimated arrival time in the global users dictionary
        if q_user_id in users:
            users[q_user_id]['estimated_arrival_time'] = recalculated_est_start_time

    print(f"User '{user_id}' left the queue for '{business['name']}'.")
    return f"User '{user_id}' successfully left '{business['name']}'."

### Testing the `join_queue` Function

Let's simulate a few users joining different queues to see how the function works and what estimated wait times are returned.

In [9]:
# Simulate User 1 joining Pharmacy A
message1, wait1 = join_queue("user_1", "pharmacy_A")
print(f"Result: {message1} Estimated Wait: {wait1} minutes.\n")

# Simulate User 2 joining Coffee Shop B
message2, wait2 = join_queue("user_2", "coffee_shop_B")
print(f"Result: {message2} Estimated Wait: {wait2} minutes.\n")

# Simulate User 3 joining Pharmacy A (after User 1)
message3, wait3 = join_queue("user_3", "pharmacy_A")
print(f"Result: {message3} Estimated Wait: {wait3} minutes.\n")

# Simulate User 4 joining a non-existent business (for error handling)
message4, wait4 = join_queue("user_4", "non_existent_business")
print(f"Result: {message4} Estimated Wait: {wait4}\n")

# Display the updated businesses and users dictionaries to see the queue changes
print("\n--- Current Business Queues ---")
for business_id, data in businesses.items():
    print(f"{data['name']} (ID: {business_id}): Queue Length = {len(data['current_queue'])}")
    for user_data in data['current_queue']:
        print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

print("\n--- Current Users ---")
for user_id, data in users.items():
    print(f"User {user_id}: Business = {data['current_business_id']}, Est. Arrival = {data['estimated_arrival_time'].strftime('%H:%M:%S') if data['estimated_arrival_time'] else 'N/A'}")

User 'user_1' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 0 minutes.
Result: Successfully joined 'QuickRx Pharmacy'. Estimated Wait: 0 minutes.

User 'user_2' joined the queue for 'Morning Brew'. Estimated wait: 0 minutes.
Result: Successfully joined 'Morning Brew'. Estimated Wait: 0 minutes.

User 'user_3' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 0 minutes.
Result: Successfully joined 'QuickRx Pharmacy'. Estimated Wait: 0 minutes.

Result: Error: Business 'non_existent_business' not found. Estimated Wait: None


--- Current Business Queues ---
QuickRx Pharmacy (ID: pharmacy_A): Queue Length = 2
  - User: user_1, Joined: 01:44:57, Est. Start: 01:44:57
  - User: user_3, Joined: 01:44:57, Est. Start: 01:44:57
Morning Brew (ID: coffee_shop_B): Queue Length = 1
  - User: user_2, Joined: 01:44:57, Est. Start: 01:44:57
Wellness Clinic (ID: clinic_C): Queue Length = 0
Classic Cuts (ID: barber_D): Queue Length = 0

--- Current Users ---
User user_1: Business = phar

### Simulating Multiple Random Users Joining Queues

Let's run a larger simulation to see how the queues evolve with multiple random users joining.

In [10]:
# Define a list of potential user IDs
all_user_ids = [f"user_{i}" for i in range(1, 11)] # 10 users

# Define a list of available business IDs
available_business_ids = list(businesses.keys())

num_simulations = 15 # Number of times users will attempt to join a queue

print(f"\n--- Starting {num_simulations} Random Queue Joins ---")
for i in range(num_simulations):
    random_user = random.choice(all_user_ids)
    random_business = random.choice(available_business_ids)

    print(f"\nSimulation {i+1}: User '{random_user}' attempts to join '{businesses[random_business]['name']}'")
    message, wait_time = join_queue(random_user, random_business)
    if wait_time is not None:
        print(f"  -> {message} Estimated Wait: {wait_time} minutes.")
    else:
        print(f"  -> {message}")

print("\n--- Simulation Complete ---")

# Display the final updated businesses and users dictionaries
print("\n--- Final Current Business Queues ---")
for business_id, data in businesses.items():
    print(f"{data['name']} (ID: {business_id}): Queue Length = {len(data['current_queue'])}")
    for user_data in data['current_queue']:
        print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

print("\n--- Final Current Users ---")
for user_id, data in users.items():
    print(f"User {user_id}: Business = {data['current_business_id']}, Est. Arrival = {data['estimated_arrival_time'].strftime('%H:%M:%S') if data['estimated_arrival_time'] else 'N/A'}")


--- Starting 15 Random Queue Joins ---

Simulation 1: User 'user_3' attempts to join 'Wellness Clinic'
  -> User 'user_3' is already in the queue for another business: 'QuickRx Pharmacy'.

Simulation 2: User 'user_2' attempts to join 'Morning Brew'
  -> User 'user_2' is already in the queue for 'Morning Brew'.

Simulation 3: User 'user_6' attempts to join 'Wellness Clinic'
User 'user_6' joined the queue for 'Wellness Clinic'. Estimated wait: 0 minutes.
  -> Successfully joined 'Wellness Clinic'. Estimated Wait: 0 minutes.

Simulation 4: User 'user_6' attempts to join 'QuickRx Pharmacy'
  -> User 'user_6' is already in the queue for another business: 'Wellness Clinic'.

Simulation 5: User 'user_7' attempts to join 'Wellness Clinic'
User 'user_7' joined the queue for 'Wellness Clinic'. Estimated wait: 15 minutes.
  -> Successfully joined 'Wellness Clinic'. Estimated Wait: 15 minutes.

Simulation 6: User 'user_7' attempts to join 'Wellness Clinic'
  -> User 'user_7' is already in the que

### Testing the `leave_queue` Function

Now let's simulate users leaving queues to observe the changes.

In [13]:
# To ensure a fresh state for testing, let's first clear the current queues and users
businesses["pharmacy_A"]["current_queue"] = []
businesses["coffee_shop_B"]["current_queue"] = []
businesses["clinic_C"]["current_queue"] = []
businesses["barber_D"]["current_queue"] = []
users = {}

print("\n--- Setting up users for Leave Queue Test ---")
join_queue("user_1_test", "pharmacy_A")
join_queue("user_2_test", "coffee_shop_B")
join_queue("user_3_test", "pharmacy_A") # This user should have a wait time
join_queue("user_4_test", "clinic_C")
join_queue("user_5_test", "barber_D")

print("\n--- Initial State Before Users Leave ---")
# Display the initial state of businesses and users for context
print("\n--- Current Business Queues (Initial) ---")
for business_id, data in businesses.items():
    print(f"{data['name']} (ID: {business_id}): Queue Length = {len(data['current_queue'])}")
    for user_data in data['current_queue']:
        print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

print("\n--- Current Users (Initial) ---")
for user_id, data in users.items():
    print(f"User {user_id}: Business = {data['current_business_id']}, Est. Arrival = {data['estimated_arrival_time'].strftime('%H:%M:%S') if data['estimated_arrival_time'] else 'N/A'}")


print("\n--- Simulating Users Leaving Queues ---")

# User 1_test leaves pharmacy_A
print(f"\nAttempting to have user_1_test leave pharmacy_A...")
message_leave_1 = leave_queue("user_1_test", "pharmacy_A")
print(f"Result: {message_leave_1}")

# User 4_test leaves clinic_C
print(f"\nAttempting to have user_4_test leave clinic_C...")
message_leave_2 = leave_queue("user_4_test", "clinic_C")
print(f"Result: {message_leave_2}")

# User 3_test leaves pharmacy_A (should be still in queue)
print(f"\nAttempting to have user_3_test leave pharmacy_A...")
message_leave_3 = leave_queue("user_3_test", "pharmacy_A")
print(f"Result: {message_leave_3}")

# Attempt to have a user leave a queue they are not in
print(f"\nAttempting to have user_1_test leave pharmacy_A again (not in queue)...")
message_leave_4 = leave_queue("user_1_test", "pharmacy_A")
print(f"Result: {message_leave_4}")

# Attempt to have a user leave a non-existent business
print(f"\nAttempting to have user_2_test leave a non-existent business...")
message_leave_5 = leave_queue("user_2_test", "non_existent_business")
print(f"Result: {message_leave_5}")

print("\n--- Final State After Users Leave ---")
# Display the updated businesses and users dictionaries
print("\n--- Current Business Queues (Final) ---")
for business_id, data in businesses.items():
    print(f"{data['name']} (ID: {business_id}): Queue Length = {len(data['current_queue'])}")
    for user_data in data['current_queue']:
        print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

print("\n--- Current Users (Final) ---")
for user_id, data in users.items():
    print(f"User {user_id}: Business = {data['current_business_id']}, Est. Arrival = {data['estimated_arrival_time'].strftime('%H:%M:%S') if data['estimated_arrival_time'] else 'N/A'}")


--- Setting up users for Leave Queue Test ---
User 'user_1_test' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 0 minutes.
User 'user_2_test' joined the queue for 'Morning Brew'. Estimated wait: 0 minutes.
User 'user_3_test' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 0 minutes.
User 'user_4_test' joined the queue for 'Wellness Clinic'. Estimated wait: 0 minutes.
User 'user_5_test' joined the queue for 'Classic Cuts'. Estimated wait: 0 minutes.

--- Initial State Before Users Leave ---

--- Current Business Queues (Initial) ---
QuickRx Pharmacy (ID: pharmacy_A): Queue Length = 2
  - User: user_1_test, Joined: 01:48:23, Est. Start: 01:48:23
  - User: user_3_test, Joined: 01:48:23, Est. Start: 01:48:23
Morning Brew (ID: coffee_shop_B): Queue Length = 1
  - User: user_2_test, Joined: 01:48:23, Est. Start: 01:48:23
Wellness Clinic (ID: clinic_C): Queue Length = 1
  - User: user_4_test, Joined: 01:48:23, Est. Start: 01:48:23
Classic Cuts (ID: barber_D): Queue Length 

### Function to Periodically Update Wait Times

To simulate a dynamic system, we need a function that periodically recalculates and updates the estimated service start times for all users currently in a business's queue. This function will consider the current time and the progress of the queue.

In [14]:
def update_wait_times(business_id):
    """
    Recalculates and updates estimated service start times for all users in a business's queue.
    This simulates the dynamic nature of wait times as people are served.
    """
    business = businesses.get(business_id)
    if not business:
        return f"Error: Business '{business_id}' not found."

    current_time = datetime.now()
    service_capacity = business['service_capacity']
    avg_customer_time = business['avg_customer_time']

    # Sort the queue by join time to ensure correct processing order
    business['current_queue'].sort(key=lambda x: x[1])

    for i, (user_id, join_time, _) in enumerate(business['current_queue']):
        if service_capacity > 0:
            # Calculate position in terms of service slots relative to the front of the queue
            queue_position_multiplier = math.floor(i / service_capacity)

            # Estimated time for the *i-th* person to start service from the *current_time*
            # We assume people are being served from the front of the queue.
            estimated_service_start_time = current_time + timedelta(minutes=queue_position_multiplier * avg_customer_time)
        else:
            estimated_service_start_time = current_time + timedelta(minutes=float('inf'))

        # Update the estimated service start time in the business's queue
        business['current_queue'][i] = (user_id, join_time, estimated_service_start_time)

        # Update the user's estimated arrival time in the global users dictionary
        if user_id in users:
            users[user_id]['estimated_arrival_time'] = estimated_service_start_time

    business['last_queue_update'] = current_time # Record when the queue was last updated
    return f"Wait times for '{business['name']}' updated at {current_time.strftime('%H:%M:%S')}."


In [18]:
def complete_service(business_id):
    """
    Simulates the completion of service for the first user in a business's queue.
    This removes the user and updates the estimated service start times for remaining users.
    """
    business = businesses.get(business_id)
    if not business:
        return f"Error: Business '{business_id}' not found."

    if not business['current_queue']:
        return f"No users in queue for '{business['name']}' to complete service."

    # Get the user who is completing service (the first in line)
    completed_user_id, _, _ = business['current_queue'].pop(0)

    # Update user information to reflect they are no longer in a queue
    if completed_user_id in users:
        users[completed_user_id]['current_business_id'] = None
        users[completed_user_id]['estimated_arrival_time'] = None

    # Recalculate estimated service start times for remaining users in this queue
    service_capacity = business['service_capacity']
    avg_customer_time = business['avg_customer_time']
    current_time = datetime.now() # Base recalculation on current time

    for i, (q_user_id, join_time, _) in enumerate(business['current_queue']):
        if service_capacity > 0:
            queue_position_multiplier = math.floor(i / service_capacity)
            recalculated_est_start_time = current_time + timedelta(minutes=queue_position_multiplier * avg_customer_time)
        else:
            recalculated_est_start_time = current_time + timedelta(minutes=float('inf'))
        business['current_queue'][i] = (q_user_id, join_time, recalculated_est_start_time)
        # Also update the user's estimated arrival time in the global users dictionary
        if q_user_id in users:
            users[q_user_id]['estimated_arrival_time'] = recalculated_est_start_time

    print(f"User '{completed_user_id}' completed service at '{business['name']}'.")
    return f"Service completed for '{completed_user_id}' at '{business['name']}'."


### Demonstration of `complete_service`

Let's demonstrate how a user completing service affects the queue and the estimated wait times for others.

In [19]:
print("\n--- Demonstrating Service Completion ---")

# Clear existing data for a clean demonstration
businesses["pharmacy_A"]["current_queue"] = []
businesses["clinic_C"]["current_queue"] = []
users = {}

# Add several users to 'clinic_C' (capacity 1, avg time 15 min)
print("\n--- Initial Joins for Clinic C ---")
join_queue("svc_user_1", "clinic_C") # Est. Wait: 0
join_queue("svc_user_2", "clinic_C") # Est. Wait: 15
join_queue("svc_user_3", "clinic_C") # Est. Wait: 30
join_queue("svc_user_4", "clinic_C") # Est. Wait: 45

print("\n--- Current Business Queues (Clinic C, Initial) ---")
for user_data in businesses["clinic_C"]['current_queue']:
    print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

# Simulate User 1 completing service
print("\n--- Simulating svc_user_1 completing service ---")
complete_message_1 = complete_service("clinic_C")
print(complete_message_1)

print("\n--- Current Business Queues (Clinic C, After svc_user_1 leaves) ---")
# Update wait times to ensure current_time is factored in for remaining users
update_wait_times("clinic_C")
for user_data in businesses["clinic_C"]['current_queue']:
    print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

# Simulate User 2 completing service
print("\n--- Simulating svc_user_2 completing service ---")
complete_message_2 = complete_service("clinic_C")
print(complete_message_2)

print("\n--- Current Business Queues (Clinic C, After svc_user_2 leaves) ---")
# Update wait times for clarity
update_wait_times("clinic_C")
for user_data in businesses["clinic_C"]['current_queue']:
    print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

print("\n--- Final User States ---")
for user_id, data in users.items():
    if user_id.startswith('svc_user_'):
        print(f"User {user_id}: Business = {data['current_business_id']}, Est. Arrival = {data['estimated_arrival_time'].strftime('%H:%M:%S') if data['estimated_arrival_time'] else 'N/A'}")

# Test edge cases: trying to complete service from an empty queue
print("\n--- Testing empty queue service completion ---")
complete_message_empty = complete_service("clinic_C")
print(complete_message_empty)

# Test edge cases: trying to complete service for non-existent business
print("\n--- Testing non-existent business service completion ---")
complete_message_nonexistent = complete_service("non_existent_business")
print(complete_message_nonexistent)


--- Demonstrating Service Completion ---

--- Initial Joins for Clinic C ---
User 'svc_user_1' joined the queue for 'Wellness Clinic'. Estimated wait: 0 minutes.
User 'svc_user_2' joined the queue for 'Wellness Clinic'. Estimated wait: 15 minutes.
User 'svc_user_3' joined the queue for 'Wellness Clinic'. Estimated wait: 30 minutes.
User 'svc_user_4' joined the queue for 'Wellness Clinic'. Estimated wait: 45 minutes.

--- Current Business Queues (Clinic C, Initial) ---
  - User: svc_user_1, Joined: 01:58:30, Est. Start: 01:58:30
  - User: svc_user_2, Joined: 01:58:30, Est. Start: 02:13:30
  - User: svc_user_3, Joined: 01:58:30, Est. Start: 02:28:30
  - User: svc_user_4, Joined: 01:58:30, Est. Start: 02:43:30

--- Simulating svc_user_1 completing service ---
User 'svc_user_1' completed service at 'Wellness Clinic'.
Service completed for 'svc_user_1' at 'Wellness Clinic'.

--- Current Business Queues (Clinic C, After svc_user_1 leaves) ---
  - User: svc_user_2, Joined: 01:58:30, Est. Sta

### Demonstration of `update_wait_times`

Let's put some users into a queue, simulate a short passage of time, and then update their wait times to see how the estimates change.

In [15]:
print("\n--- Demonstrating Wait Time Updates ---")

# Clear existing data for a clean demonstration
businesses["pharmacy_A"]["current_queue"] = []
businesses["coffee_shop_B"]["current_queue"] = []
users = {}

# Add some users to 'pharmacy_A'
print("\n--- Initial Joins ---")
join_queue("update_user_A", "pharmacy_A") # Capacity: 2, Avg Time: 8 min
join_queue("update_user_B", "pharmacy_A")
join_queue("update_user_C", "pharmacy_A")
join_queue("update_user_D", "pharmacy_A")
join_queue("update_user_E", "pharmacy_A")

print("\n--- Current Business Queues (Initial) ---")
for business_id, data in businesses.items():
    if business_id == 'pharmacy_A': # Only show pharmacy_A for this demo
        print(f"{data['name']} (ID: {business_id}): Queue Length = {len(data['current_queue'])}")
        for user_data in data['current_queue']:
            print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

# Simulate passage of time (e.g., 5 minutes)
print("\nSimulating 5 minutes passing...")
time.sleep(5) # In a real scenario, this would be actual time elapsed

# Update wait times for pharmacy_A
print("\n--- Updating Wait Times for Pharmacy A ---")
update_message = update_wait_times("pharmacy_A")
print(update_message)

print("\n--- Current Business Queues (After Update) ---")
for business_id, data in businesses.items():
    if business_id == 'pharmacy_A': # Only show pharmacy_A for this demo
        print(f"{data['name']} (ID: {business_id}): Queue Length = {len(data['current_queue'])}")
        for user_data in data['current_queue']:
            print(f"  - User: {user_data[0]}, Joined: {user_data[1].strftime('%H:%M:%S')}, Est. Start: {user_data[2].strftime('%H:%M:%S')}")

print("\n--- Current Users (After Update) ---")
for user_id, data in users.items():
    if user_id.startswith('update_user_'): # Only show demo users
        print(f"User {user_id}: Business = {data['current_business_id']}, Est. Arrival = {data['estimated_arrival_time'].strftime('%H:%M:%S') if data['estimated_arrival_time'] else 'N/A'}")


--- Demonstrating Wait Time Updates ---

--- Initial Joins ---
User 'update_user_A' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 0 minutes.
User 'update_user_B' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 0 minutes.
User 'update_user_C' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 4 minutes.
User 'update_user_D' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 8 minutes.
User 'update_user_E' joined the queue for 'QuickRx Pharmacy'. Estimated wait: 12 minutes.

--- Current Business Queues (Initial) ---
QuickRx Pharmacy (ID: pharmacy_A): Queue Length = 5
  - User: update_user_A, Joined: 01:54:55, Est. Start: 01:54:55
  - User: update_user_B, Joined: 01:54:55, Est. Start: 01:54:55
  - User: update_user_C, Joined: 01:54:55, Est. Start: 02:02:55
  - User: update_user_D, Joined: 01:54:55, Est. Start: 02:02:55
  - User: update_user_E, Joined: 01:54:55, Est. Start: 02:10:55

Simulating 5 minutes passing...

--- Updating Wait Times for Pharmacy A --

### Function to Report Business Statistics

This function will provide an overview of the current state of all businesses in the system, including their queue lengths and estimated wait times.

In [16]:
def report_business_statistics():
    """
    Generates and prints a report of current statistics for all businesses.
    """
    print("\n--- Business Statistics Report ---")
    if not businesses:
        print("No businesses found in the system.")
        return

    for business_id, data in businesses.items():
        name = data['name']
        queue_length = len(data['current_queue'])
        # Calculate estimated wait time for a new customer joining now
        estimated_wait = calculate_estimated_wait_time(business_id)

        print(f"\nBusiness: {name} (ID: {business_id})")
        print(f"  - Current Queue Length: {queue_length} users")
        print(f"  - Estimated Wait for New Customer: {estimated_wait} minutes")
        if queue_length > 0:
            print(f"  - Users in Queue:")
            for user_id, join_time, est_start_time in data['current_queue']:
                print(f"    - User: {user_id}, Joined: {join_time.strftime('%H:%M:%S')}, Est. Start: {est_start_time.strftime('%H:%M:%S')}")


### Demonstration of `report_business_statistics`

Let's run the new function to see the current state of all businesses after our previous simulations.

In [17]:
report_business_statistics()


--- Business Statistics Report ---

Business: QuickRx Pharmacy (ID: pharmacy_A)
  - Current Queue Length: 5 users
  - Estimated Wait for New Customer: 12 minutes
  - Users in Queue:
    - User: update_user_A, Joined: 01:54:55, Est. Start: 01:55:00
    - User: update_user_B, Joined: 01:54:55, Est. Start: 01:55:00
    - User: update_user_C, Joined: 01:54:55, Est. Start: 02:03:00
    - User: update_user_D, Joined: 01:54:55, Est. Start: 02:03:00
    - User: update_user_E, Joined: 01:54:55, Est. Start: 02:11:00

Business: Morning Brew (ID: coffee_shop_B)
  - Current Queue Length: 0 users
  - Estimated Wait for New Customer: 0 minutes

Business: Wellness Clinic (ID: clinic_C)
  - Current Queue Length: 0 users
  - Estimated Wait for New Customer: 0 minutes

Business: Classic Cuts (ID: barber_D)
  - Current Queue Length: 1 users
  - Estimated Wait for New Customer: 0 minutes
  - Users in Queue:
    - User: user_5_test, Joined: 01:48:23, Est. Start: 01:48:23
